<a href="https://colab.research.google.com/github/crystalclcm/Dissertation-Crystal-Matticks/blob/main/Post_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import requests
import pandas as pd

In [2]:
APP_ID = "90fb93ab"
APP_KEY = "ac0d74800544bfc63431a2d3747a92c5"


In [3]:
import requests
import pandas as pd
import time
import random

APP_ID = "90fb93ab"
APP_KEY = "ac0d74800544bfc63431a2d3747a92c5"

BASE_URL = "https://api.adzuna.com/v1/api/jobs/{country}/search/{page}"

roles = [
    "Accountant",
    "Financial Analyst",
    "Business Analyst",
    "Risk Analyst",
    "Software Engineer",
    "Data Analyst",
    "Data Engineer",
    "Cybersecurity Analyst"
]

countries = ["gb", "de", "nl", "us", "ca", "au"]  # UK, Germany, Netherlands, US, Canada, Australia

def fetch_adzuna_jobs(country, role, pages=10, results_per_page=50):
    all_results = []

    for page in range(1, pages + 1):
        print(f"Country: {country.upper()} | Role: {role} | Page: {page}")

        params = {
            "app_id": APP_ID,
            "app_key": APP_KEY,
            "results_per_page": results_per_page,
            "what": role,
            "content-type": "application/json"
        }

        url = BASE_URL.format(country=country, page=page)

        try:
            r = requests.get(url, params=params, timeout=15)
            if r.status_code != 200:
                print(f" Status code {r.status_code} for {country} - {role} - page {page}")
                continue

            data = r.json()
            jobs = data.get("results", [])

            for job in jobs:
                all_results.append({
                    "role": role,
                    "country": country.upper(),
                    "job_title": job.get("title"),
                    "company": job.get("company", {}).get("display_name"),
                    "location": job.get("location", {}).get("display_name"),
                    "created": job.get("created"),
                    "redirect_url": job.get("redirect_url"),
                    "description": job.get("description")
                })

        except Exception as e:
            print(f" Error for {country} - {role} - page {page}: {e}")

        time.sleep(random.uniform(1.0, 2.5))

    return all_results

all_results = []

for country in countries:
    for role in roles:
        results = fetch_adzuna_jobs(country, role, pages=10, results_per_page=50)
        all_results.extend(results)

df = pd.DataFrame(all_results)
print("Total rows scraped:", len(df))

df.to_csv("adzuna_expanded_combined.csv", index=False)
df.head()


Country: GB | Role: Accountant | Page: 1
Country: GB | Role: Accountant | Page: 2
Country: GB | Role: Accountant | Page: 3
Country: GB | Role: Accountant | Page: 4
Country: GB | Role: Accountant | Page: 5
Country: GB | Role: Accountant | Page: 6
Country: GB | Role: Accountant | Page: 7
Country: GB | Role: Accountant | Page: 8
Country: GB | Role: Accountant | Page: 9
Country: GB | Role: Accountant | Page: 10
Country: GB | Role: Financial Analyst | Page: 1
Country: GB | Role: Financial Analyst | Page: 2
Country: GB | Role: Financial Analyst | Page: 3
Country: GB | Role: Financial Analyst | Page: 4
Country: GB | Role: Financial Analyst | Page: 5
Country: GB | Role: Financial Analyst | Page: 6
Country: GB | Role: Financial Analyst | Page: 7
Country: GB | Role: Financial Analyst | Page: 8
Country: GB | Role: Financial Analyst | Page: 9
Country: GB | Role: Financial Analyst | Page: 10
Country: GB | Role: Business Analyst | Page: 1
Country: GB | Role: Business Analyst | Page: 2
Country: GB | 

,role,country,job_title,company,location,created,redirect_url,description
0,Accountant,GB,Accountant - Accounts preparation,HAYS,"Belfast, Northern Ireland",2026-02-24T10:27:17Z,https://www.adzuna.co.uk/jobs/land/ad/56434395...,"Accounts Preperation, Accountant, Accounts You..."
1,Accountant,GB,Accounts Manager (Qualified Accountant),HAYS,"Norwich, Norfolk",2026-02-24T10:27:25Z,https://www.adzuna.co.uk/jobs/land/ad/56434402...,ACA/ACCA Accounts Manager job in Norwich – Div...
2,Accountant,GB,Management Accountant,2 sisters Food Group,"Tiverton Junction, Cullompton",2026-04-13T19:43:13Z,https://www.adzuna.co.uk/jobs/land/ad/56982899...,Management Accountant Location: Willand Workin...
3,Accountant,GB,Accountant - Accounts Senior,SI Recruitment,"Bottesford, Scunthorpe",2026-03-27T11:39:10Z,https://www.adzuna.co.uk/jobs/land/ad/56803023...,Experienced Accountant / Accounts Senior Pract...
4,Accountant,GB,Account Executive,Currys,"Derby, Derbyshire",2026-04-14T01:56:14Z,https://www.adzuna.co.uk/jobs/land/ad/56985000...,"Role overview: Account Executive Derby Currys,..."


In [4]:
df['role'].value_counts()
df['country'].value_counts()


,count
country,
US,4000
GB,3398
CA,3179
AU,3004
DE,2694
NL,2540


In [5]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [6]:
df.to_csv('/content/drive/MyDrive/adzuna_expanded_combined.csv', index=False)


In [7]:
import os
os.makedirs("data/post_ai/adzuna", exist_ok=True)


In [8]:
import requests
import pandas as pd

APP_ID = "90fb93ab"
APP_KEY = "ac0d74800544bfc63431a2d3747a92c5"

def get_adzuna_jobs(country_code, role):
    url = f"https://api.adzuna.com/v1/api/jobs/{country_code}/search/1"
    params = {
        "app_id": APP_ID,
        "app_key": APP_KEY,
        "results_per_page": 50,
        "what": role
    }

    response = requests.get(url, params=params).json()
    results = response.get("results", [])

    rows = []
    for job in results:
        rows.append({
            "job_title": job.get("title"),
            "job_description": job.get("description"),
            "country": country_code,
            "date_posted": job.get("created"),
            "source": "adzuna"
        })

    return pd.DataFrame(rows)


In [9]:
df = get_adzuna_jobs("gb", "accountant")
df.to_csv("data/post_ai/adzuna/accountant_gb.csv", index=False)

df = get_adzuna_jobs("de", "accountant")
df.to_csv("data/post_ai/adzuna/accountant_de.csv", index=False)

df = get_adzuna_jobs("nl", "accountant")
df.to_csv("data/post_ai/adzuna/accountant_nl.csv", index=False)


In [10]:
df = get_adzuna_jobs("gb", "business analyst")
df.to_csv("data/post_ai/adzuna/business_analyst_gb.csv", index=False)

df = get_adzuna_jobs("de", "business analyst")
df.to_csv("data/post_ai/adzuna/business_analyst_de.csv", index=False)

df = get_adzuna_jobs("nl", "business analyst")
df.to_csv("data/post_ai/adzuna/business_analyst_nl.csv", index=False)


In [11]:
df = get_adzuna_jobs("gb", "data analyst")
df.to_csv("data/post_ai/adzuna/data_analyst_gb.csv", index=False)

df = get_adzuna_jobs("de", "data analyst")
df.to_csv("data/post_ai/adzuna/data_analyst_de.csv", index=False)

df = get_adzuna_jobs("nl", "data analyst")
df.to_csv("data/post_ai/adzuna/data_analyst_nl.csv", index=False)


In [12]:
df = get_adzuna_jobs("gb", "software engineer")
df.to_csv("data/post_ai/adzuna/software_engineer_gb.csv", index=False)

df = get_adzuna_jobs("de", "software engineer")
df.to_csv("data/post_ai/adzuna/software_engineer_de.csv", index=False)

df = get_adzuna_jobs("nl", "software engineer")
df.to_csv("data/post_ai/adzuna/software_engineer_nl.csv", index=False)


# cleaning

In [13]:
import pandas as pd

def clean_adzuna(df):
    df = df.rename(columns={
        "title": "job_title",
        "job_title": "job_title",
        "description": "job_description",
        "job_description": "job_description",
        "created": "date_posted",
        "country": "country"
    })

    df["source"] = "adzuna"

    keep = ["job_title", "job_description", "country", "date_posted", "source"]
    df = df[[c for c in keep if c in df.columns]]

    return df


In [14]:
df = pd.read_csv("data/post_ai/adzuna/accountant_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_accountant_gb.csv", index=False)


In [15]:
df = pd.read_csv("data/post_ai/adzuna/accountant_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_accountant_de.csv", index=False)


In [16]:
df = pd.read_csv("data/post_ai/adzuna/accountant_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_accountant_nl.csv", index=False)


In [17]:
df = pd.read_csv("data/post_ai/adzuna/business_analyst_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_business_analyst_gb.csv", index=False)


In [18]:
df = pd.read_csv("data/post_ai/adzuna/business_analyst_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_business_analyst_de.csv", index=False)


In [19]:
df = pd.read_csv("data/post_ai/adzuna/business_analyst_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_business_analyst_nl.csv", index=False)

In [20]:
df = pd.read_csv("data/post_ai/adzuna/data_analyst_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_data_analyst_gb.csv", index=False)


In [21]:
df = pd.read_csv("data/post_ai/adzuna/data_analyst_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_data_analyst_de.csv", index=False)


In [22]:
df = pd.read_csv("data/post_ai/adzuna/data_analyst_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_data_analyst_nl.csv", index=False)


In [23]:
df = pd.read_csv("data/post_ai/adzuna/software_engineer_gb.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_software_engineer_gb.csv", index=False)


In [24]:
df = pd.read_csv("data/post_ai/adzuna/software_engineer_de.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_software_engineer_de.csv", index=False)


In [25]:
df = pd.read_csv("data/post_ai/adzuna/software_engineer_nl.csv")
cleaned = clean_adzuna(df)
cleaned.to_csv("data/post_ai/adzuna/cleaned_software_engineer_nl.csv", index=False)


In [26]:
import glob

files = glob.glob("data/post_ai/adzuna/*")
files


['data/post_ai/adzuna/cleaned_data_analyst_nl.csv',
 'data/post_ai/adzuna/data_analyst_gb.csv',
 'data/post_ai/adzuna/software_engineer_nl.csv',
 'data/post_ai/adzuna/cleaned_accountant_gb.csv',
 'data/post_ai/adzuna/data_analyst_nl.csv',
 'data/post_ai/adzuna/software_engineer_de.csv',
 'data/post_ai/adzuna/accountant_de.csv',
 'data/post_ai/adzuna/cleaned_business_analyst_gb.csv',
 'data/post_ai/adzuna/cleaned_data_analyst_de.csv',
 'data/post_ai/adzuna/accountant_gb.csv',
 'data/post_ai/adzuna/cleaned_business_analyst_nl.csv',
 'data/post_ai/adzuna/cleaned_business_analyst_de.csv',
 'data/post_ai/adzuna/data_analyst_de.csv',
 'data/post_ai/adzuna/cleaned_software_engineer_gb.csv',
 'data/post_ai/adzuna/cleaned_data_analyst_gb.csv',
 'data/post_ai/adzuna/business_analyst_gb.csv',
 'data/post_ai/adzuna/cleaned_software_engineer_nl.csv',
 'data/post_ai/adzuna/cleaned_accountant_nl.csv',
 'data/post_ai/adzuna/accountant_nl.csv',
 'data/post_ai/adzuna/business_analyst_de.csv',
 'data/pos

In [27]:
import pandas as pd
import glob

cleaned_files = glob.glob("data/post_ai/adzuna/cleaned_*.csv")

for f in cleaned_files:
    print(f"\n--- Preview of {f} ---")
    df = pd.read_csv(f)
    print(df.head(5))



--- Preview of data/post_ai/adzuna/cleaned_data_analyst_nl.csv ---
                                     job_title  \
0                                 Data analist   
1  Senior Data Analyst (Business Intelligence)   
2  Senior Data Analyst (Business Intelligence)   
3                          Junior Data Analist   
4                                 Data Analist   

                                     job_description country  \
0  Wil jij met veel afwisseling als data analist ...      nl   
1  Vacature kenmerken Standplaats Huis ter Heide ...      nl   
2  Vacature kenmerken Standplaats Huis ter Heide ...      nl   
3  Sta je aan het begin van je carrière en wil je...      nl   
4  Amsterdam Hybride Functieomschrijving Werk je ...      nl   

            date_posted  source  
0  2026-04-08T18:41:25Z  adzuna  
1  2026-04-10T19:06:11Z  adzuna  
2  2026-04-10T19:10:57Z  adzuna  
3  2025-12-16T10:55:41Z  adzuna  
4  2026-02-28T01:54:03Z  adzuna  

--- Preview of data/post_ai/adzuna/cleane

In [28]:
df = pd.read_csv("data/post_ai/adzuna/cleaned_data_analyst_gb.csv")
df.head().T


,0,1,2,3,4
job_title,Data Analyst Trainee,Data Analyst Trainee,Trainee Data Analyst Associate,Trainee Data Analyst - job guarantee,Data Analyst
job_description,Are you looking to benefit from a new career i...,Are you ready to start a new career in Data An...,Are you ready to start a new career in Data An...,Are you ready to start a new career in Data An...,"Data Analyst | Chesterfield | Starting at £26,..."
country,gb,gb,gb,gb,gb
date_posted,2026-04-02T13:51:52Z,2026-04-08T04:29:39Z,2026-04-08T04:29:38Z,2026-04-08T04:29:38Z,2026-03-17T19:59:20Z
source,adzuna,adzuna,adzuna,adzuna,adzuna


In [29]:
import pandas as pd
import glob

# find all cleaned files
cleaned_files = glob.glob("data/post_ai/adzuna/cleaned_*.csv")

# load them into a list of DataFrames
dfs = [pd.read_csv(f) for f in cleaned_files]

# merge into one dataset
adzuna_post_ai = pd.concat(dfs, ignore_index=True)

# save the merged dataset
adzuna_post_ai.to_csv("data/post_ai/adzuna/adzuna_post_ai_merged.csv", index=False)

adzuna_post_ai.head()


,job_title,job_description,country,date_posted,source
0,Data analist,Wil jij met veel afwisseling als data analist ...,nl,2026-04-08T18:41:25Z,adzuna
1,Senior Data Analyst (Business Intelligence),Vacature kenmerken Standplaats Huis ter Heide ...,nl,2026-04-10T19:06:11Z,adzuna
2,Senior Data Analyst (Business Intelligence),Vacature kenmerken Standplaats Huis ter Heide ...,nl,2026-04-10T19:10:57Z,adzuna
3,Junior Data Analist,Sta je aan het begin van je carrière en wil je...,nl,2025-12-16T10:55:41Z,adzuna
4,Data Analist,Amsterdam Hybride Functieomschrijving Werk je ...,nl,2026-02-28T01:54:03Z,adzuna


In [30]:
adzuna_post_ai.shape


(600, 5)

In [31]:
import pandas as pd

# load your merged dataset
df = pd.read_csv("data/post_ai/adzuna/adzuna_post_ai_merged.csv")

def assign_role(title):
    title_lower = str(title).lower()
    if "accountant" in title_lower:
        return "accountant"
    if "business analyst" in title_lower:
        return "business_analyst"
    if "data analyst" in title_lower:
        return "data_analyst"
    if "software engineer" in title_lower or "developer" in title_lower:
        return "software_engineer"
    return "unknown"

df["role"] = df["job_title"].apply(assign_role)

df.to_csv("data/post_ai/adzuna/adzuna_post_ai_with_roles.csv", index=False)

df.head(150)


,job_title,job_description,country,date_posted,source,role
0,Data analist,Wil jij met veel afwisseling als data analist ...,nl,2026-04-08T18:41:25Z,adzuna,unknown
1,Senior Data Analyst (Business Intelligence),Vacature kenmerken Standplaats Huis ter Heide ...,nl,2026-04-10T19:06:11Z,adzuna,data_analyst
2,Senior Data Analyst (Business Intelligence),Vacature kenmerken Standplaats Huis ter Heide ...,nl,2026-04-10T19:10:57Z,adzuna,data_analyst
3,Junior Data Analist,Sta je aan het begin van je carrière en wil je...,nl,2025-12-16T10:55:41Z,adzuna,unknown
4,Data Analist,Amsterdam Hybride Functieomschrijving Werk je ...,nl,2026-02-28T01:54:03Z,adzuna,unknown
...,...,...,...,...,...,...
145,Technical Business Analyst,Your new company An international commercial l...,gb,2026-03-25T15:53:42Z,adzuna,business_analyst
146,Business Analyst,We are looking for a Business Analyst to join ...,gb,2026-04-10T23:55:22Z,adzuna,business_analyst
147,Business Analyst,"Do you enjoy analysing problems, working with ...",gb,2026-04-09T13:57:31Z,adzuna,business_analyst
148,Business Analyst,"Do you enjoy analysing problems, working with ...",gb,2026-04-09T14:08:24Z,adzuna,business_analyst


In [32]:
df = pd.read_csv("data/post_ai/adzuna/adzuna_post_ai_with_roles.csv")
df["role"].value_counts()


,count
role,
software_engineer,146
accountant,139
business_analyst,118
data_analyst,116
unknown,81


In [33]:
df[df["role"] == "unknown"][["job_title"]].head(30)


,job_title
0,Data analist
3,Junior Data Analist
4,Data Analist
5,Data Analist
7,Data Analist
8,Data Analist
9,Data Analist
10,Data Analist
11,Data Analist
12,Data Analist


In [34]:
def assign_role(title):
    t = str(title).lower()

    # Accountant
    if "accountant" in t or "accounts assistant" in t or "finance assistant" in t:
        return "accountant"

    # Business Analyst
    if "business analyst" in t or "ba " in t or "business intelligence analyst" in t:
        return "business_analyst"

    # Data Analyst
    if (
        "data analyst" in t or
        "data analist" in t or
        "data-analist" in t or
        "data governance" in t or
        "data operations" in t or
        "reporting analyst" in t or
        "bi analyst" in t or
        "analytics" in t
    ):
        return "data_analyst"

    # Software Engineer / Developer / Data Engineer
    if (
        "software engineer" in t or
        "developer" in t or
        "engineer" in t or
        "data engineer" in t or
        "full stack" in t or
        "backend" in t or
        "frontend" in t
    ):
        return "software_engineer"

    return "unknown"


In [35]:
df["role"] = df["job_title"].apply(assign_role)
df["role"].value_counts()


,count
role,
data_analyst,147
software_engineer,147
accountant,139
business_analyst,118
unknown,49


In [36]:
df[df["role"] == "unknown"]["job_title"].sample(20, random_state=42)


,job_title
201,Teamleider Business Analisten bij IT-Infra
275,Senior Business Systems Analyst (m/w/d) (m/w/d)
495,Manager accountancy & advies in Rotterdam
273,Business Application Analyst (m/w/d) für SAP
208,Business Analist
225,Business Analist
224,Business Analist
220,Business Analist
229,Business Analist
210,Business Analist


In [37]:
def assign_role(title):
    t = str(title).lower()

    # Accountant (English + Dutch)
    if (
        "accountant" in t or
        "boekhoudkundig" in t or
        "accounts assistant" in t or
        "finance assistant" in t
    ):
        return "accountant"

    # Business Analyst (English + Dutch + German + variants)
    if (
        "business analyst" in t or
        "business analist" in t or
        "business-analist" in t or
        "business intelligence analist" in t or
        "business intelligence analyst" in t or
        "business it-analyst" in t or
        "business systems analyst" in t or
        "business applications" in t or
        "market analyst" in t or
        "analyst prozessdigitalisierung" in t or
        "business & market analyst" in t
    ):
        return "business_analyst"

    # Data Analyst (English + Dutch + German + variants)
    if (
        "data analyst" in t or
        "data analist" in t or
        "data-analist" in t or
        "data governance" in t or
        "data operations" in t or
        "reporting analyst" in t or
        "bi analyst" in t or
        "analytics" in t
    ):
        return "data_analyst"

    # Software Engineer / Developer / Data Engineer (English + German)
    if (
        "software engineer" in t or
        "developer" in t or
        "entwickler" in t or
        "engineer" in t or
        "data engineer" in t or
        "full stack" in t or
        "backend" in t or
        "frontend" in t
    ):
        return "software_engineer"

    return "unknown"


In [38]:
df["role"] = df["job_title"].apply(assign_role)
df["role"].value_counts()


,count
role,
business_analyst,149
data_analyst,147
software_engineer,147
accountant,139
unknown,18


In [39]:
df = df[df["role"] != "unknown"]
df.to_csv("data/post_ai/adzuna/adzuna_post_ai_final.csv", index=False)


In [44]:
!cp data/post_ai/adzuna/adzuna_post_ai_final.csv /content/drive/MyDrive/adzuna_post_ai_final.csv


In [48]:
from google.colab import files
files.download('/content/drive/MyDrive/adzuna_post_ai_final.csv')


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [41]:
import os
os.listdir('/content/drive/MyDrive')


['Easilet Lease Control Sheet.xlsx',
 'Easilet Lease Control Sheet.gsheet',
 'Car Spaces  Byrne Moore.xls',
 'Car Spaces  Byrne Moore.gsheet',
 'BW Contact Listing.xlsx',
 'BW Contact Listing.gsheet',
 'RH Customer listing.xlsx',
 'Parking control listing.xlsx',
 'Parking control listing.gsheet',
 'RH Customer listing.gsheet',
 'GLS Customer Listing.xlsx',
 'GLS Customer Listing.gsheet',
 'BM Listing.xlsx',
 'BM Listing.gsheet',
 'GCV Supplier Listing.xlsx',
 'GCV Supplier Listing.gsheet',
 'lease of unit 5, bymac(part 1).pdf',
 '30A BW lease agreement (second half).pdf',
 '30A BW lease agreement (first half).pdf',
 'lease of unit 5, bymac(part 2).pdf',
 '6 ferndale road letter of engagement.pdf',
 '36 st. fintan villas letter of engagement.pdf',
 '36 st. fintan villas lease agreement (part 1).pdf',
 '36 st.fintan villas lease agreement (part 2).pdf',
 '68 st. fintan villas letter of engagement.pdf',
 'index.pdf',
 'Document31 (2).pdf',
 'Document31 (2).gdoc',
 'Colab Notebooks',
 'Unt